In [1]:
import site
import sys
import subprocess
import os

# 1. Registrar la ruta de paquetes del usuario en el sistema
user_site = '/headless/.local/lib/python3.12/site-packages'
if os.path.exists(user_site):
    site.addsitedir(user_site)

# 2. Intentar importar; si falla, instalarlo automáticamente
try:
    from PyLTSpice import RawRead
    print('¡PyLTSpice ya estaba instalado y listo para usar!')
except ImportError:
    print('PyLTSpice no fue encontrado. Instalando automáticamente...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'PyLTSpice'])
    if os.path.exists(user_site):
        site.addsitedir(user_site)
    from PyLTSpice import RawRead
    print('¡PyLTSpice se instaló e importó correctamente!')


¡PyLTSpice ya estaba instalado y listo para usar!


In [2]:
import os
import re
import subprocess
import shutil
import concurrent.futures
import itertools
from PyLTSpice import RawRead
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

parametros_ldo = {}
graficas_ldo = {}

def one_pvt_parallel(netlist_path, raw_path, temp, vdd, proces):
    # Nombre de la carpeta limpio y seguro
    vdd_str = str(vdd).replace('.', '_')
    nombre_llave = f"V_{vdd_str}_T_{temp}_{proces}"
    
    parametros_ldo[nombre_llave] = {}
    graficas_ldo[nombre_llave] = {}
    
    # 1. CREAR LA CARPETA AISLADA PARA ESTE HILO
    sim_dir = os.path.join(raw_path, nombre_llave)
    os.makedirs(sim_dir, exist_ok=True)
    
    directorio_original = os.path.dirname(netlist_path)
    
    with open(netlist_path, 'r') as archivo:
        contenido = archivo.read()

    # Modificar parámetros PVT
    patron_corner = r"(\.lib\s+/foss/pdks/sky130A/libs\.tech/ngspice/sky130\.lib\.spice\s+)\w+"
    contenido = re.sub(patron_corner, rf"\g<1>{proces}", contenido)
    
    patron_temp = r"(\.option\s+temp=)[-\d\.]+"
    contenido = re.sub(patron_temp, rf"\g<1>{temp}", contenido)
    
    patron_vdd = r"(\.param\s+VDD\s*=\s*)[-\d\.]+"
    contenido = re.sub(patron_vdd, rf"\g<1>{vdd}", contenido)
    
    # ====================================================================
    # 2. REDIRIGIR LA SALIDA A LA CARPETA AISLADA
    # ====================================================================
    nombres_base = [
        "LDO_tran_picos",
        "LDO_ac_estability",
        "LDO_ac_psrr",
        "LDO_dc_line",      # Ahora es transitorio, pero mantenemos el nombre .raw
        "LDO_dc_load",      # Ahora es transitorio, pero mantenemos el nombre .raw
        "LDO_dc_dropout"    # Agregado para el dropout
    ]
    
    for base in nombres_base:
        ruta_absoluta_raw = os.path.join(sim_dir, f"{base}.raw")
        contenido = contenido.replace(f"{base}.raw", ruta_absoluta_raw)
    
    # Guardar netlist temporal en el directorio original para que lea los .include
    netlist_temp_path = os.path.join(directorio_original, f"ldo_temp_{nombre_llave}.spice")
    with open(netlist_temp_path, 'w') as archivo:
        archivo.write(contenido)

    # 3. SIMULAR
    comando = ['/foss/tools/ngspice/bin/ngspice', "-b", netlist_temp_path]
    resultado = subprocess.run(comando, capture_output=True, text=True, cwd=directorio_original)
    
    if resultado.returncode != 0:
        if os.path.exists(netlist_temp_path):
            os.remove(netlist_temp_path)
        return

    # Extraer LOG de variables de consola
    mi_log = resultado.stdout
    patron = r"([a-zA-Z0-9_]+)\s*=\s*([+-]?\d+\.?\d*[eE]?[+-]?\d*)"
    coincidencias = re.findall(patron, mi_log)
    
    for nombre, valor in coincidencias:
        parametros_ldo[nombre_llave][nombre.strip().lower()] = float(valor)
    
    if "tnom" in parametros_ldo[nombre_llave]:
        del parametros_ldo[nombre_llave]["tnom"]

    # 4. EXTRAER LAS GRÁFICAS DESDE LOS ARCHIVOS RAW (PyLTSpice)
    try:
        raw_tran = RawRead(os.path.join(sim_dir, "LDO_tran_picos.raw"))
        raw_ac_stability = RawRead(os.path.join(sim_dir, "LDO_ac_estability.raw"))
        raw_ac_psrr = RawRead(os.path.join(sim_dir, "LDO_ac_psrr.raw"))
        raw_dc_line = RawRead(os.path.join(sim_dir, "LDO_dc_line.raw"))
        raw_dc_load = RawRead(os.path.join(sim_dir, "LDO_dc_load.raw"))
        
        # Picos
        graficas_ldo[nombre_llave]["time"] = raw_tran.get_trace("time").get_wave()
        graficas_ldo[nombre_llave]["t_vo"] = raw_tran.get_trace("v(vo)").get_wave()
        
        # Estabilidad
        graficas_ldo[nombre_llave]["frecuencia"] = raw_ac_stability.get_trace("frequency").get_wave()
        graficas_ldo[nombre_llave]["fase"] = raw_ac_stability.get_trace("fase").get_wave()
        graficas_ldo[nombre_llave]["ganancia"] = raw_ac_stability.get_trace("ldb").get_wave()
        
        # PSRR
        graficas_ldo[nombre_llave]["psrr"] = raw_ac_psrr.get_trace("psrr_db").get_wave()
        
        # --- CAMBIO A TRANSIENTE PARA LINE REGULATION ---
        graficas_ldo[nombre_llave]["line_time"] = raw_dc_line.get_trace("time").get_wave()
        graficas_ldo[nombre_llave]["line_vo"] = raw_dc_line.get_trace("v(vo)").get_wave()
        
        # --- CAMBIO A TRANSIENTE PARA LOAD REGULATION ---
        graficas_ldo[nombre_llave]["load_time"] = raw_dc_load.get_trace("time").get_wave()
        graficas_ldo[nombre_llave]["load_vo"] = raw_dc_load.get_trace("v(vo)").get_wave()
        
    except Exception as e:
        print(f"\n⚠️ Error al leer los datos de {nombre_llave}: {e}")

    # ====================================================================
    # 5. LIMPIEZA
    # ====================================================================
    if os.path.exists(netlist_temp_path):
        os.remove(netlist_temp_path)
        
    # shutil.rmtree(sim_dir, ignore_errors=True) # Descomentar para borrar los raws



In [3]:

# ==============================================================================
# 3. LANZADOR MULTINÚCLEO (PARALELISMO)
# ==============================================================================
temperaturas = [-40, 27, 125] #[-40, 27, 125]         # Temperaturas en grados Celsius [°C]
voltajes = [1.62, 1.80, 1.98]         # Niveles de alimentación VDD [V]  [1.62, 1.80, 1.9]

# Esquinas de proceso estándar principales
corners_proceso = [
    "tt", "sf", "ff", "ss", "fs", "ll", "hh", "hl", "lh",
    "sf_ll", "sf_hh", "sf_hl", "sf_lh", "ff_ll", "ff_hh", "ff_hl", "ff_lh",
    "ss_ll", "ss_hh", "ss_hl", "ss_lh", "fs_ll", "fs_hh", "fs_hl", "fs_lh"
]


"""
[
    "tt", "sf", "ff", "ss", "fs", "ll", "hh", "hl", "lh",
    "sf_ll", "sf_hh", "sf_hl", "sf_lh", "ff_ll", "ff_hh", "ff_hl", "ff_lh",
    "ss_ll", "ss_hh", "ss_hl", "ss_lh", "fs_ll", "fs_hh", "fs_hl", "fs_lh"
]

corners_criticos = [
    "ss", "ff", "sf", "fs", "ss_hh"
"""

netlist_path = "/headless/.xschem/simulations/ldo_tb.spice"
raw_path = "/foss/designs/LDO-sky130/xschem/spice_raw_log/raw/"

todas_las_combinaciones = list(itertools.product(temperaturas, voltajes, corners_proceso))
total_sim = len(todas_las_combinaciones)

print(f"Lanzando {total_sim} simulaciones en PARALELO...")

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futuros = [
        executor.submit(one_pvt_parallel, netlist_path, raw_path, temp, vdd, proces) 
        for temp, vdd, proces in todas_las_combinaciones
    ]
    
    completados = 0
    for futuro in concurrent.futures.as_completed(futuros):
        completados += 1
        print(f"Progreso global: {completados}/{total_sim} completados.", end='\r')

print("\n\n¡Simulación masiva paralela finalizada con éxito!")

# ==============================================================================
# 4. REPORTES NUMÉRICOS Y EXCEL
# ==============================================================================
df_resultados = pd.DataFrame.from_dict(parametros_ldo, orient='index')
df_resumen = df_resultados.agg(['min', 'max', 'mean']).T

print("--- Resumen PVT: Peor y Mejor Caso de cada Especificación ---")
try:
    display(df_resumen)
except NameError:
    print(df_resumen)

try:
    with pd.ExcelWriter("Reporte_Completo_LDO.xlsx") as writer:
        df_resultados.to_excel(writer, sheet_name="225_Corners")
        df_resumen.to_excel(writer, sheet_name="Resumen_Peores_Casos")
    print("\n¡Reporte exportado exitosamente a 'Reporte_Completo_LDO.xlsx'!")
    
except ImportError:
    print("\nLibrería de Excel no detectada. Guardando como CSV...")
    df_resultados.to_csv("225_Corners_LDO.csv")
    df_resumen.to_csv("Resumen_Peores_Casos_LDO.csv")
    print("¡Tablas exportadas exitosamente a formato CSV!")


Lanzando 225 simulaciones en PARALELO...
Progreso global: 225/225 completados.

¡Simulación masiva paralela finalizada con éxito!


ValueError: No objects to concatenate

In [4]:
print("\nLibrería de Excel no detectada. Guardando como CSV...")
df_resultados.to_csv("30_Corners_LDO.csv")
df_resumen.to_csv("Resumen_Peores_Casos_LDO.csv")
print("¡Tablas exportadas exitosamente a formato CSV!")


Librería de Excel no detectada. Guardando como CSV...


NameError: name 'df_resumen' is not defined

In [5]:
# ====================================================================
# 5. GRAFICADOR DE TODAS LAS CURVAS (CORREGIDO PARA VECTORES INCOMPLETOS)
# ====================================================================
print("\nGenerando gráficas...")

# --- 1. PSRR ---
plt.figure(figsize=(10, 6))
exitosas_psrr = 0
for llave, curvas in graficas_ldo.items():
    if "frecuencia" in curvas and "psrr" in curvas:
        x = np.abs(curvas["frecuencia"])
        y = np.real(curvas["psrr"])
        
        # Igualar tamaños al mínimo disponible
        min_len = min(len(x), len(y))
        
        plt.semilogx(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_psrr += 1
        
plt.title(f"Análisis PVT: PSRR del LDO (Curvas: {exitosas_psrr})")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("PSRR [dB]")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.savefig("psrr.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 2. RESPUESTA TRANSITORIA (Overshoot) ---
plt.figure(figsize=(10, 6))
exitosas_tran = 0
for llave, curvas in graficas_ldo.items():
    if "time" in curvas and "t_vo" in curvas:
        x = np.real(curvas["time"])
        y = np.real(curvas["t_vo"])
        y2 = np.real(curvas["t_vo"])
        
        min_len = min(len(x), len(y))
        
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_tran += 1
        
plt.title(f"Análisis PVT: Respuesta Transitoria (Curvas: {exitosas_tran})")
plt.xlabel("Tiempo [s]")
plt.ylabel("Voltaje de Salida (Vo) [V]")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.savefig("tran.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 3. ESTABILIDAD ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
exitosas_estab = 0
for llave, curvas in graficas_ldo.items():
    if "frecuencia" in curvas and "ganancia" in curvas and "fase" in curvas:
        x = np.abs(curvas["frecuencia"])
        y_gain = np.real(curvas["ganancia"])
        y_phase = np.real(curvas["fase"])
        
        min_len = min(len(x), len(y_gain), len(y_phase))
        
        ax1.semilogx(x[:min_len], y_gain[:min_len], linewidth=0.8, alpha=0.3)
        ax2.semilogx(x[:min_len], y_phase[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_estab += 1

ax1.set_title(f"Análisis PVT: Estabilidad Lazo Abierto (Curvas: {exitosas_estab})")
ax1.set_ylabel("Ganancia (Loop Gain) [dB]")
ax1.grid(True, which="both", linestyle="--", linewidth=0.5)
ax1.axhline(0, color='red', linestyle='-', linewidth=1.5, label="0 dB")
ax1.legend()
ax2.set_xlabel("Frecuencia [Hz]")
ax2.set_ylabel("Fase [Grados]")
ax2.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig("estabilidad.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 4. REGULACIÓN DE LÍNEA (Rampa) ---
plt.figure(figsize=(10, 6))
exitosas_line = 0
for llave, curvas in graficas_ldo.items():
    if "line_time" in curvas and "line_vo" in curvas:
        x = np.real(curvas["line_time"])
        y = np.real(curvas["line_vo"])
        
        min_len = min(len(x), len(y))
        
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_line += 1
        
plt.title(f"Análisis PVT: Regulación de Línea (Curvas: {exitosas_line})")
plt.xlabel("Tiempo de Rampa [s] (Simula Vin)")
plt.ylabel("Voltaje de Salida (Vo) [V]")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.savefig("grafica_regulacion_linea.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 5. REGULACIÓN DE CARGA (Rampa) ---
plt.figure(figsize=(10, 6))
exitosas_load = 0
for llave, curvas in graficas_ldo.items():
    if "load_time" in curvas and "load_vo" in curvas:
        x = np.real(curvas["load_time"])
        y = np.real(curvas["load_vo"])
        
        min_len = min(len(x), len(y))
        
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_load += 1
        
plt.title(f"Análisis PVT: Regulación de Carga (Curvas: {exitosas_load})")
plt.xlabel("Tiempo de Rampa [s] (Simula Iload)")
plt.ylabel("Voltaje de Salida (Vo) [V]")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.savefig("grafica_regulacion_carga.png", dpi=300, bbox_inches='tight')
plt.close()

print("Todas las gráficas se guardaron en la carpeta actual.")


Generando gráficas...
Todas las gráficas se guardaron en la carpeta actual.


In [6]:
import pickle
import os

# ... (Todo tu código anterior y la ejecución de concurrent.futures) ...

# Asegúrate de que este bloque vaya DESPUÉS de que terminen todos los procesos en paralelo
def guardar_resultados():
    carpeta_destino = "resultados_simulacion"
    os.makedirs(carpeta_destino, exist_ok=True)
    
    # Guardar los parámetros (escalares)
    ruta_parametros = os.path.join(carpeta_destino, "parametros_ldo_1cargamr_12_5.pkl")
    with open(ruta_parametros, 'wb') as archivo:
        pickle.dump(parametros_ldo, archivo)
        
    # Guardar los datos de las gráficas (arreglos de numpy)
    ruta_graficas = os.path.join(carpeta_destino, "graficas_ldo_1carga_mr12_5.pkl")
    with open(ruta_graficas, 'wb') as archivo:
        pickle.dump(graficas_ldo, archivo)
        
    print(f"? Datos guardados exitosamente en la carpeta: {carpeta_destino}")

# Llamar a la función al finalizar las simulaciones
guardar_resultados()

? Datos guardados exitosamente en la carpeta: resultados_simulacion
